In [ ]:
# пайплайн S3 (MinIO) → ClickHouse в Jupyter Notebook
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, current_date

#pip install pyspark jupyterlab --break-system-packages

#создаёт Spark-сессию с автоматической загрузкой JAR-зависимостей для работы с внешними системами
spark = SparkSession.builder \
    .appName("s3_final") \
    .config("spark.jars.packages", 
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.adaptive.enabled", "false") \
    .getOrCreate()
"""    
spark = SparkSession.builder \
    .appName("SparkExample") \
    .config("spark.jars.packages",
        "org.apache.hadoop:hadoop-aws:3.3.4,"
        "com.amazonaws:aws-java-sdk-bundle:1.12.262,"
        "ru.yandex.clickhouse:clickhouse-jdbc:0.3.2,"
        "org.postgresql:postgresql:42.5.0,"
        "org.apache.spark:spark-sql-kafka-0-10_2.12:3.3.0",
        ) \
    .getOrCreate()
"""

In [ ]:

# MinIO/S3 настройки
hadoop_conf = spark._jsc.hadoopConfiguration()

hadoop_conf.set("fs.s3a.access.key", "minioadmin")
hadoop_conf.set("fs.s3a.secret.key", "minioadmin")
hadoop_conf.set("fs.s3a.endpoint", "http://minio:9000")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

#
hadoop_conf.set("fs.s3a.connection.timeout", "60000")
hadoop_conf.set("fs.s3a.connection.establish.timeout", "30000")
hadoop_conf.set("fs.s3a.connection.idle.time", "60000")
hadoop_conf.set("fs.s3a.connection.request.timeout", "60000")
hadoop_conf.set("fs.s3a.connection.ttl", "300000")
hadoop_conf.set("fs.s3a.attempts.maximum", "5")


# подключение к s3 и путь сохранения
MINIO_CONN_ID = 'minios3_conn'
TELEGRAM_USER = 'shushiato'
BUCKET_NAME = 'dev'


In [ ]:
#rm -rf /tmp/spark-streaming-*  # Локальные checkpoints
#docker exec -it minio mc ls dev/shushiato/thespacedevs_1.parquet
#spark.sparkContext._jvm.org.apache.hadoop.fs.FileSystem.getDefaultUri()
#spark._jsc.hadoopConfiguration().get("fs.defaultFS")
try:
    df = spark.read.parquet("s3a://dev/shushiato/thespacedevs_1.parquet")
    print("✅ S3 работает!")
    df.show(3)
except Exception as e:
    print(f"❌ Ошибка: {e}")

In [ ]:
# Чтение Parquet из S3
#s3_hook = S3Hook(aws_conn_id=MINIO_CONN_ID)
df = spark.read.parquet("s3a://dev/shushiato/thespacedevs_1.parquet")
df.show(3, truncate=False)
print(f"✅ {df.count()} строк загружено!")

In [ ]:

spark.stop()